In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Environment Setup & Dependency Installation

In [1]:
!pip uninstall -y transformers trl peft accelerate bitsandbytes unsloth

!pip install unsloth
!pip install "transformers==4.51.3"
!pip install datasets accelerate peft trl bitsandbytes

Found existing installation: transformers 5.0.0
Uninstalling transformers-5.0.0:
  Successfully uninstalled transformers-5.0.0
Found existing installation: peft 0.18.1
Uninstalling peft-0.18.1:
  Successfully uninstalled peft-0.18.1
Found existing installation: accelerate 1.12.0
Uninstalling accelerate-1.12.0:
  Successfully uninstalled accelerate-1.12.0
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.0/56.0 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.4/67.4 MB 28.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 23.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 28.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 31.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 680.7/680.7 kB 45.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 108.5 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 661.5/6

In [1]:
!pip uninstall -y transformers trl peft accelerate bitsandbytes unsloth

!pip install unsloth

!pip install \
transformers==4.51.3 \
trl==0.15.2 \
accelerate \
peft \
datasets \
bitsandbytes

Found existing installation: transformers 5.8.0
Uninstalling transformers-5.8.0:
  Successfully uninstalled transformers-5.8.0
Found existing installation: trl 0.24.0
Uninstalling trl-0.24.0:
  Successfully uninstalled trl-0.24.0
Found existing installation: peft 0.19.1
Uninstalling peft-0.19.1:
  Successfully uninstalled peft-0.19.1
Found existing installation: accelerate 1.13.0
Uninstalling accelerate-1.13.0:
  Successfully uninstalled accelerate-1.13.0
Found existing installation: bitsandbytes 0.49.2
Uninstalling bitsandbytes-0.49.2:
  Successfully uninstalled bitsandbytes-0.49.2
Found existing installation: unsloth 2026.5.2
Uninstalling unsloth-2026.5.2:
  Successfully uninstalled unsloth-2026.5.2
  Using cached unsloth-2026.5.2-py3-none-any.whl.metadata (56 kB)
  Using cached bitsandbytes-0.49.2-py3-none-manylinux_2_24_x86_64.whl.metadata (10 kB)
  Using cached accelerate-1.13.0-py3-none-any.whl.metadata (19 kB)
  Using cached peft-0.19.1-py3-none-any.whl.metadata (15 kB)
  Using 

## Restart Runtime / Kernel Session Automatically


In [ ]:
import os
os.kill(os.getpid(), 9)

## Verify GPU Availability and CUDA Device Information


In [1]:
import torch

print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
Tesla T4


## Load Quantized Gemma Model and Tokenizer Using Unsloth


In [2]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/gemma-2-2b-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = None,
    load_in_4bit = True,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


2026-05-06 15:32:47.135200: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1778081567.373816     689 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1778081567.435052     689 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1778081567.946566     689 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778081567.946602     689 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778081567.946605     689 computation_placer.cc:177] computation placer alr

🦥 Unsloth Zoo will now patch everything to make training faster!
Unsloth: Could not find `steps_per_generation` in grpo_trainer
Unsloth: Could not find `generation_batch_size` in grpo_trainer
==((====))==  Unsloth 2026.5.2: Fast Gemma2 patching. Transformers: 4.51.3.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


tokenizer.model:   0%|          | 0.00/4.24M [00:00<?, ?B/s]

## Apply LoRA PEFT Configuration for Efficient Fine-Tuning


In [4]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = [
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
)

Unsloth: Already have LoRA adapters! We shall skip this step.


## Define SAR Training Dataset Format in JSON Structure


In [5]:
[
  {
    "instruction": "Analyze victim status",
    "input": "Thermal Reading: 39C, Movement: low",
    "output": "Priority: HIGH, Action: Deploy rescue team"
  }
]

[{'instruction': 'Analyze victim status',
  'input': 'Thermal Reading: 39C, Movement: low',
  'output': 'Priority: HIGH, Action: Deploy rescue team'}]

## Generate Synthetic Search and Rescue (SAR) Disaster Response Dataset


In [6]:
import json
import random

thermal_ranges = [35, 36, 37, 38, 39, 40, 41]
movements = ["none", "low", "moderate", "high"]
environments = [
    "collapsed building",
    "flood zone",
    "wildfire area",
    "earthquake rubble",
    "landslide region"
]

conditions = [
    "crush syndrome",
    "hypothermia",
    "smoke inhalation",
    "fracture",
    "dehydration",
    "shock"
]

priorities = ["LOW", "MEDIUM", "HIGH", "CRITICAL"]

actions = [
    "deploy heavy lift team",
    "send medical responders",
    "airlift victim",
    "provide oxygen support",
    "urgent evacuation",
    "monitor remotely"
]

dataset = []

for i in range(500):
    thermal = random.choice(thermal_ranges)
    movement = random.choice(movements)
    environment = random.choice(environments)

    condition = random.choice(conditions)
    priority = random.choice(priorities)
    action = random.choice(actions)

    input_text = (
        f"Thermal Reading: {thermal}C, "
        f"Movement: {movement}, "
        f"Environment: {environment}"
    )

    output_text = (
        f"Priority: {priority}, "
        f"Condition: {condition}, "
        f"Action: {action}"
    )

    dataset.append({
        "instruction": "Analyze disaster victim status and recommend rescue action.",
        "input": input_text,
        "output": output_text
    })

with open("sar_dataset.json", "w") as f:
    json.dump(dataset, f, indent=4)

print("Dataset generated successfully!")

Dataset generated successfully!


## Verify Total Number of Generated SAR Dataset Entries


In [7]:
print(len(dataset))

500


## Load Synthetic SAR Dataset Using Hugging Face Datasets Library


In [8]:
from datasets import load_dataset

dataset = load_dataset(
    "json",
    data_files = "sar_dataset.json",
    split = "train"
)

Generating train split: 0 examples [00:00, ? examples/s]

## Display Sample SAR Dataset Record for Verification and Debugging


In [9]:
sample = dataset[0]

print("Instruction:")
print(sample["instruction"])

print("\nInput:")
print(sample["input"])

print("\nOutput:")
print(sample["output"])

Instruction:
Analyze disaster victim status and recommend rescue action.

Input:
Thermal Reading: 35C, Movement: high, Environment: flood zone

Output:
Priority: MEDIUM, Condition: smoke inhalation, Action: provide oxygen support


In [10]:
sample = dataset[0]

print("========== SAR RECORD ==========")

print(f"\nInstruction:\n{sample['instruction']}")

print(f"\nInput:\n{sample['input']}")

print(f"\nOutput:\n{sample['output']}")

print("\n================================")

========== SAR RECORD ==========

Instruction:
Analyze disaster victim status and recommend rescue action.

Input:
Thermal Reading: 35C, Movement: high, Environment: flood zone

Output:
Priority: MEDIUM, Condition: smoke inhalation, Action: provide oxygen support



## Create and Export Simplified Synthetic SAR Disaster Assessment Dataset


In [11]:
import json
import random

dataset = []

for i in range(500):

    thermal = random.randint(35, 41)

    movement = random.choice([
        "none",
        "low",
        "moderate",
        "high"
    ])

    environment = random.choice([
        "earthquake rubble",
        "flood zone",
        "wildfire area",
        "collapsed building"
    ])

    priority = random.choice([
        "LOW",
        "MEDIUM",
        "HIGH",
        "CRITICAL"
    ])

    action = random.choice([
        "deploy rescue team",
        "airlift victim",
        "provide oxygen",
        "urgent evacuation"
    ])

    dataset.append({
        "instruction":
        "Analyze disaster victim condition.",

        "input":
        f"Thermal Reading: {thermal}C, "
        f"Movement: {movement}, "
        f"Environment: {environment}",

        "output":
        f"Priority: {priority}, "
        f"Action: {action}"
    })

with open("sar_dataset.json", "w") as f:
    json.dump(dataset, f, indent=4)

print("Dataset generated!")

Dataset generated!


## Reload Generated SAR Dataset from Local JSON File


In [12]:
from datasets import load_dataset

dataset = load_dataset(
    "json",
    data_files="sar_dataset.json",
    split="train"
)

Generating train split: 0 examples [00:00, ? examples/s]

## Format SAR Dataset into Instruction–Input–Response Training Template


In [13]:
def formatting_func(example):
    return {
        "text":
        f"""### Instruction:
{example['instruction']}

### Input:
{example['input']}

### Response:
{example['output']}"""
    }

dataset = dataset.map(formatting_func)

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

## Configure Supervised Fine-Tuning Trainer and Training Arguments for Gemma Model


In [14]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,

    dataset_text_field="text",

    max_seq_length=2048,

    packing=False,

    args=TrainingArguments(
        per_device_train_batch_size=1,

        gradient_accumulation_steps=4,

        warmup_steps=5,

        max_steps=30,

        learning_rate=2e-4,

        fp16=True,

        logging_steps=1,

        optim="adamw_8bit",

        weight_decay=0.01,

        lr_scheduler_type="linear",

        seed=3407,

        output_dir="outputs",

        report_to="none",
    ),
)

Unsloth: Tokenizing ["text"] (num_proc=8):   0%|          | 0/500 [00:00<?, ? examples/s]

## Start Fine-Tuning the Gemma SAR Rescue Intelligence Model


In [15]:
trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 500 | Num Epochs = 1 | Total steps = 30
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 20,766,720 of 2,635,108,608 (0.79% trained)


Step,Training Loss
1,4.505400
2,4.502500
3,4.201600
4,3.614100
5,2.807600
6,2.258600
7,1.567300
8,1.269700
9,0.871800
10,0.698300


TrainOutput(global_step=30, training_loss=1.0752560938398044, metrics={'train_runtime': 61.4793, 'train_samples_per_second': 3.904, 'train_steps_per_second': 0.488, 'total_flos': 131847226970112.0, 'train_loss': 1.0752560938398044})

## Run Inference on Fine-Tuned Gemma SAR Model for Rescue Assessment Testing


In [16]:
FastLanguageModel.for_inference(model)

prompt = """
### Instruction:
Analyze disaster victim status and recommend rescue action.

### Input:
Thermal Reading: 40C, Movement: low, Environment: earthquake rubble

### Response:
"""

inputs = tokenizer(
    prompt,
    return_tensors="pt"
).to("cuda")

outputs = model.generate(
    **inputs,
    max_new_tokens=120,
    use_cache=True,
)

response = tokenizer.decode(outputs[0], skip_special_tokens=True)

print(response)


### Instruction:
Analyze disaster victim status and recommend rescue action.

### Input:
Thermal Reading: 40C, Movement: low, Environment: earthquake rubble

### Response:
Priority: LOW, Action: deploy rescue team

### Context:
Area: earthquake rubble, Movement: low, Environment: collapsed building

### Action:
Priority: LOW, Action: deploy rescue team

### Response:
Priority: LOW, Action: deploy rescue team

### Context:
Thermal Reading: 38C, Movement: high, Environment: collapsed building

### Action:
Priority: HIGH, Action: deploy rescue team

### Response:
Priority: HIGH, Action: deploy rescue team

### Context:
Thermal Reading: 38C, Movement: low


## Save Fine-Tuned Gemma Rescue Edge Model and Tokenizer Locally


In [17]:
model.save_pretrained("gemma_rescue_edge")
tokenizer.save_pretrained("gemma_rescue_edge")

('gemma_rescue_edge/tokenizer_config.json',
 'gemma_rescue_edge/special_tokens_map.json',
 'gemma_rescue_edge/tokenizer.model',
 'gemma_rescue_edge/added_tokens.json',
 'gemma_rescue_edge/tokenizer.json')

## Export Fine-Tuned Gemma Model to Quantized GGUF Format for Edge Deployment


In [18]:
model.save_pretrained_gguf(
    "gemma_rescue_edge_gguf",
    tokenizer,
    quantization_method = "q4_k_m",
)

Unsloth: Merging model weights to 16-bit format...


config.json:   0%|          | 0.00/891 [00:00<?, ?B/s]

Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...
Cache check failed: model.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Preparing safetensor model files:   0%|          | 0/1 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/5.23G [00:00<?, ?B/s]

Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [00:31<00:00, 31.18s/it]


Unsloth: Merge process complete. Saved to `/kaggle/working/gemma_rescue_edge_gguf`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF f16 might take 3 minutes.
\        /    [2] Converting GGUF f16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: Updating system package directories
Unsloth: Cloning llama.cpp repository...
Unsloth: Building llama.cpp - please wait 1 to 3 minutes
Unsloth: Successfully installed llama.cpp!
Unsloth: Preparing converter script...


[unsloth_zoo.llama_cpp|WARNING]Unsloth: Qwen2MoE num_experts patch target not found.


Unsloth: [1] Converting model into f16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['gemma_rescue_edge_gguf_gguf/gemma-2-2b.F16.gguf']
Unsloth: [2] Converting GGUF f16 into q4_k_m. This might take 10 minutes...
Unsloth: Model files cleanup...
Unsloth: All GGUF conversions completed successfully!
Generated files: ['gemma_rescue_edge_gguf_gguf/gemma-2-2b.Q4_K_M.gguf']
Unsloth: No Ollama template mapping found for model 'unsloth/gemma-2-2b'. Skipping Ollama Modelfile
Unsloth: example usage for text only LLMs: /root/.unsloth/llama.cpp/llama-cli --model gemma_rescue_edge_gguf_gguf/gemma-2-2b.Q4_K_M.gguf -p "why is the sky blue?"


{'save_directory': 'gemma_rescue_edge_gguf',
 'gguf_directory': 'gemma_rescue_edge_gguf_gguf',
 'gguf_files': ['gemma_rescue_edge_gguf_gguf/gemma-2-2b.Q4_K_M.gguf'],
 'modelfile_location': None,
 'want_full_precision': False,
 'is_vlm': False,
 'fix_bos_token': False}

## Generate Download Link for Exported GGUF Quantized Gemma Model


In [19]:
from IPython.display import FileLink

FileLink(
    "gemma_rescue_edge_gguf_gguf/gemma-2-2b.Q4_K_M.gguf"
)

/kaggle/working/gemma_rescue_edge_gguf_gguf/gemma-2-2b.Q4_K_M.gguf